<a href="https://colab.research.google.com/github/nejumi/models-initial-course/blob/main/notebooks/03_core/02_tables_media.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. Tables と Media: 具体例をまとめて動かす

## この章のゴール
W&B で扱える代表的な media / data type を、実際に log して確認します。

この notebook は他の章より長めです。授業では重要なセルだけ実行し、後から「この形式を log したい」と思ったときのリファレンスとして使えます。

In [ ]:
# リポジトリ内ではルートを使い、Colab 単体では共通コードを取得します。
from pathlib import Path
import os
import sys
import urllib.request

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").is_file()), Path.cwd())
if not (root / "course_utils.py").exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/nejumi/models-initial-course/main/course_utils.py",
        root / "course_utils.py",
    )
os.chdir(root)
sys.path.insert(0, str(root))

from course_utils import setup_wandb
PROJECT, ENTITY, WANDB_INIT = setup_wandb(packages=['wandb[media]>=0.19.0', 'numpy>=1.24.0', 'pillow>=10.0.0', 'plotly>=5.20.0', 'moviepy>=1.0.3', 'imageio>=2.31.0', 'imageio-ffmpeg>=0.4.9'])

## 1. 画像: PIL / NumPy array / file path
同じ `wandb.Image` でも、渡し方を変えられます。

In [ ]:
from pathlib import Path
import numpy as np
import wandb
from course_utils import demo_image_array, demo_pil_image, save_demo_image

media_dir = Path("outputs/05_media")
media_dir.mkdir(parents=True, exist_ok=True)
image_path = save_demo_image(media_dir / "demo-image.png")

with wandb.init(**WANDB_INIT, name="05a-images", tags=["course", "media"]) as run:
    run.log({
        "image/pil": wandb.Image(demo_pil_image("PIL"), caption="PIL.Image object"),
        "image/numpy": wandb.Image(demo_image_array(), caption="NumPy RGB array"),
        "image/file_path": wandb.Image(image_path, caption="image file path"),
    })
    print(run.url)

## 2. 画像 overlay: Bounding box と Segmentation mask
物体検出やセグメンテーションの結果を画像に重ねて確認できます。

In [ ]:
from course_utils import demo_mask

class_labels = {0: "background", 1: "target A", 2: "target B"}
boxes = {
    "prediction": {
        "box_data": [
            {
                "position": {"minX": 14, "minY": 16, "maxX": 58, "maxY": 72},
                "domain": "pixel",
                "class_id": 1,
                "box_caption": "min/max box",
                "scores": {"confidence": 0.91},
            },
            {
                "position": {"middle": [72, 58], "width": 34, "height": 42},
                "domain": "pixel",
                "class_id": 2,
                "box_caption": "middle/size box",
                "scores": {"confidence": 0.66},
            },
        ],
        "class_labels": class_labels,
    }
}

with wandb.init(**WANDB_INIT, name="05b-image-overlays", tags=["course", "media"]) as run:
    run.log({
        "image/boxes": wandb.Image(demo_image_array(), boxes=boxes),
        "image/mask": wandb.Image(
            demo_image_array(),
            masks={"prediction": {"mask_data": demo_mask(), "class_labels": class_labels}},
        ),
    })
    print(run.url)

## 3. Table: text と media を同じ行に並べる
評価サンプル、予測、score、画像、音声、HTML を同じ Table に入れます。

In [ ]:
from course_utils import demo_audio_array, label_image

audio, sample_rate = demo_audio_array(freq=523.25)
table = wandb.Table(columns=["id", "input", "prediction", "score", "image", "audio", "html"])
table.add_data(
    "sample-001",
    "短い入力文",
    "ok",
    0.92,
    wandb.Image(label_image("ok", "#b7e4c7")),
    wandb.Audio(audio, sample_rate=sample_rate, caption="sine wave"),
    wandb.Html("<b>review:</b> looks good"),
)
table.add_data(
    "sample-002",
    "確認が必要な入力文",
    "needs_review",
    0.41,
    wandb.Image(label_image("needs_review", "#ffcc33")),
    wandb.Audio(audio * 0.5, sample_rate=sample_rate, caption="quieter sine wave"),
    wandb.Html("<b>review:</b> check this row"),
)

with wandb.init(**WANDB_INIT, name="05c-table-mixed-media", tags=["course", "media", "table"]) as run:
    run.log({"eval/mixed_media_table": table})
    print(run.url)

## 4. Audio: NumPy array / WAV file path
音声は `wandb.Audio` で log します。

In [ ]:
from course_utils import save_demo_wav

audio, sample_rate = demo_audio_array(freq=440.0)
wav_path = save_demo_wav(media_dir / "demo-audio.wav")

with wandb.init(**WANDB_INIT, name="05d-audio", tags=["course", "media"]) as run:
    run.log({
        "audio/from_numpy": wandb.Audio(audio, sample_rate=sample_rate, caption="NumPy array"),
        "audio/from_file": wandb.Audio(wav_path, caption="WAV file path"),
    })
    print(run.url)

## 5. Video: NumPy array / GIF file path
動画は `wandb.Video` で log します。NumPy array を渡す場合は `moviepy` / `imageio` 系の依存関係が必要です。

In [ ]:
from course_utils import demo_video_array, save_demo_gif

gif_path = save_demo_gif(media_dir / "demo-video.gif")
video_array = demo_video_array()

with wandb.init(**WANDB_INIT, name="05e-video", tags=["course", "media"]) as run:
    run.log({
        "video/from_numpy": wandb.Video(video_array, fps=4, format="gif"),
        "video/from_file": wandb.Video(gif_path, fps=4, format="gif"),
    })
    print(run.url)

## 6. HTML: string / file
任意の HTML を run に残せます。

In [ ]:
html_path = media_dir / "demo.html"
html_path.write_text("<h3>HTML file</h3><p>This HTML was logged from a file.</p>")

with wandb.init(**WANDB_INIT, name="05f-html", tags=["course", "media"]) as run:
    with open(html_path) as f:
        run.log({
            "html/from_string": wandb.Html("<h3>HTML string</h3><p>This HTML was logged from a string.</p>"),
            "html/from_file": wandb.Html(f, inject=False),
        })
    print(run.url)

## 7. 3D: Object3D point cloud / lidar scene
3D 点群や lidar 形式の scene を log できます。

In [ ]:
from course_utils import demo_point_cloud_rgb, demo_lidar_scene

scene = demo_lidar_scene()

with wandb.init(**WANDB_INIT, name="05g-object3d", tags=["course", "media"]) as run:
    run.log({
        "object3d/xyzrgb": wandb.Object3D(demo_point_cloud_rgb()),
        "object3d/lidar_scene": wandb.Object3D(scene),
    })
    print(run.url)

## 8. Molecule: PDB file / SMILES
分子構造は `wandb.Molecule` で log できます。`wandb[media]` を入れておくと SMILES 例も動かせます。

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdmolfiles
Chem.rdmolfiles = rdmolfiles  # W&B 0.31 + RDKit 2026.9 compatibility.
from course_utils import write_demo_pdb

pdb_path = write_demo_pdb(media_dir / "water.pdb")
molecule = Chem.MolFromPDBFile(pdb_path, removeHs=False)
assert molecule is not None and molecule.GetNumAtoms() == 3, "PDB の構造を確認してください"

with wandb.init(**WANDB_INIT, name="05h-molecule", tags=["course", "media"]) as run:
    run.log({
        "molecule/from_pdb": wandb.Molecule(pdb_path),
        "molecule/from_smiles": wandb.Molecule.from_smiles("CCO"),
    })
    print(run.url)

## 9. Histogram
配列を `wandb.Histogram` として log できます。

In [ ]:
rng = np.random.default_rng(0)
values = rng.normal(loc=0.0, scale=1.0, size=512)
np_histogram = np.histogram(values, bins=32)

with wandb.init(**WANDB_INIT, name="05i-histogram", tags=["course", "media"]) as run:
    run.log({
        "histogram/auto": wandb.Histogram(values),
        "histogram/from_numpy": wandb.Histogram(np_histogram=np_histogram),
    })
    print(run.url)

## 10. Plotly と Custom Charts
Plotly figure と `wandb.plot` の代表的な chart を log します。

In [ ]:
from course_utils import demo_plotly_figure

chart_table = wandb.Table(columns=["step", "loss", "group"])
for step, loss in enumerate([0.9, 0.62, 0.48, 0.39]):
    chart_table.add_data(step, loss, "run-a")

bar_table = wandb.Table(data=[["A", 0.82], ["B", 0.74], ["C", 0.91]], columns=["label", "score"])
y_true = [0, 1, 2, 1, 0, 2]
y_pred = [0, 1, 1, 1, 0, 2]

with wandb.init(**WANDB_INIT, name="05j-plots", tags=["course", "media", "plot"]) as run:
    run.log({
        "plotly/loss": wandb.Plotly(demo_plotly_figure()),
        "chart/line": wandb.plot.line(chart_table, "step", "loss", title="Loss by step"),
        "chart/scatter": wandb.plot.scatter(chart_table, "step", "loss", title="Step vs loss"),
        "chart/bar": wandb.plot.bar(bar_table, "label", "score", title="Score by label"),
        "chart/histogram": wandb.plot.histogram(chart_table, "loss", title="Loss histogram"),
        "chart/confusion_matrix": wandb.plot.confusion_matrix(
            y_true=y_true,
            preds=y_pred,
            class_names=["class_0", "class_1", "class_2"],
        ),
    })
    print(run.url)

## W&B App で確認すること
- Media panels に Image / Audio / Video / HTML / Object3D / Molecule が表示される
- Table 内で text、score、image、audio、HTML を同じ行で確認できる
- Image overlay の box / mask を切り替えられる
- Custom Charts で line、scatter、bar、histogram、confusion matrix を確認できる